Cleaning and Validating the Data

We did a little data cleaning previously, but now we are going to learn repeatable data-cleaning habits, as well as how to prepare the dataset for feature engineering. So, we're going to proceed as if we had no information from the previous notebook.

First, load the dataset.


In [7]:
import pandas as pd

df = pd.read_csv("../data/raw/books.csv")
print(f"Rows loaded: {len(df)}")
df.head()


Rows loaded: 100


,book_id,title,author,genres,publication_year,page_count,average_rating,rating_count,description
0,B001,Dune,Frank Herbert,Science Fiction;Politics;Adventure,1965,412,4.27,850000,A young nobleman becomes entangled in a strugg...
1,B002,Hyperion,Dan Simmons,Science Fiction;Space Opera;Adventure,1989,482,4.22,185000,Seven travelers journey toward a mysterious wo...
2,B003,The Left Hand of Darkness,Ursula K. Le Guin,Science Fiction;Politics;Literary,1969,304,4.08,190000,An envoy visits a distant planet and must navi...
3,B004,The Dispossessed,Ursula K. Le Guin,Science Fiction;Politics;Literary,1974,387,4.23,170000,A physicist moves between contrasting worlds w...
4,B005,Foundation,Isaac Asimov,Science Fiction;Space Opera;Politics,1951,296,4.17,500000,A mathematician predicts the collapse of a vas...


In [8]:
print(df.columns.tolist())

['book_id', 'title', 'author', 'genres', 'publication_year', 'page_count', 'average_rating', 'rating_count', 'description']


Check for any exact duplicates

In [9]:
print(df.duplicated().sum())

#this returns 0, because book_id will be different.

0


In [10]:
#looking for duplicates regardless of book_id

duplicate_mask = df.drop(columns=["book_id"]).duplicated(keep=False)
display(df.loc[duplicate_mask].sort_values("title"))

,book_id,title,author,genres,publication_year,page_count,average_rating,rating_count,description
18,B019,The Hobbit,J. R. R. Tolkien,Fantasy;Adventure;Classic,1937,310,4.29,2100000,A reluctant hobbit leaves home with a company ...
60,B061,The Hobbit,J. R. R. Tolkien,Fantasy;Adventure;Classic,1937,310,4.29,2100000,A reluctant hobbit leaves home with a company ...


In [11]:
#This keeps the first matching metadata record. For larger real-world data, we will check editions or ISBNs before merging.

df_clean = df.drop_duplicates(
    subset=df.columns.drop("book_id").tolist()
).copy()
print("Original rows:", len(df))
print("Cleaned rows:", len(df_clean))


Original rows: 100
Cleaned rows: 99


In [12]:
#removal verification
print(df_clean.drop(columns=["book_id"]).duplicated().sum())

0


Now let's check data types.
We want to check that publication_year, page_count, average_rating, and rating_count are numeric, and that title, author, genres, and description are text.

In [13]:
df_clean.dtypes

book_id                 str
title                   str
author                  str
genres                  str
publication_year      int64
page_count            int64
average_rating      float64
rating_count          int64
description             str
dtype: object

The data types are all correct, but let's proceed as if they weren't.

In [17]:
numeric_columns = [
    "publication_year", "page_count",
    "average_rating", "rating_count"
]
for column in numeric_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column], errors="coerce"
    )

#Invalid numeric values become missing values (NaN), which we will inspect rather than guess replacements for.